# Chapter 8: Image Classification - Advanced Exercises

## Datasets for Practice

**1\. The Dogs vs. Cats Dataset** This is a classic computer vision dataset containing 25,000 images of dogs and cats1,2. It is the benchmark dataset used throughout Chapter 8 to demonstrate small-data problem techniques3.

-   **How to download:** You can download it using the Kaggle API (the `kagglehub` package is preinstalled in Google Colab)4,5:-   _Note: To simulate a "small data" problem, organize a subset of this data into directories containing 2,000 training images, 1,000 validation images, and 2,000 test images_3.

**2\. The Oxford Flowers 102 Dataset** A challenging dataset containing 8,189 images of flowers categorized into 102 different species.

-   **How to download:** Use Keras to fetch and extract the archive directly7:

\--------------------------------------------------------------------------------

# Advanced Exercises

## Exercise 1: The Impact of Data Augmentation under Extreme Scarcity

Overfitting is the primary challenge when dealing with small image datasets, and data augmentation mitigates this by generating believable variations of existing training samples so the model never sees the exact same picture twice8.

-   **Dataset:** Dogs vs. Cats.-   **Task:** Restrict your training set to just **500 images per class** (1,000 total).-   **Part A:** Build a ConvNet from scratch using an alternating stack of `Conv2D` and `MaxPooling2D` layers, ending with a `GlobalAveragePooling2D` layer and a dense classifier6,9. Train it for 30 epochs and plot the loss curves to identify exactly when severe overfitting begins.-   **Part B:** Define a data augmentation pipeline using `layers.RandomFlip("horizontal")`, `layers.RandomRotation(0.1)`, and `layers.RandomZoom(0.2)`10. Map this augmentation to your `tf.data.Dataset` using `train_dataset.map(data_augmentation, num_parallel_calls=8)`10. Add a `layers.Dropout(0.25)` layer before your final classifier11. Train again for 100 epochs and compare the validation accuracy and overfitting trajectory to Part A.

In [ ]:
# Part A
# Download the dogs and cats dataset and select 500 images per class
import keras
import kagglehub
import os
import logging
import zipfile

logging.basicConfig(level=logging.INFO)


os.environ["KAGGLE_API_TOKEN"] = "KGAT_b16fca105497fc0a804ee5d19d4699e6"

kagglehub.login()  # Requires a Kaggle account/API key
download_path = kagglehub.competition_download("dogs-vs-cats")
data_directory = os.path.join(download_path, "dogs-vs-cats")
for root, dirs, files in os.walk(download_path):
    for filename in files:
        logging.info(os.path.join(root, filename))
        if filename.endswith(".zip"):
          # Unzip the dataset and log its contents
          with zipfile.ZipFile(os.path.join(root, filename), "r") as zip_ref:
            # Move the unzipped files to a specific directory if needed
            zip_ref.extractall(os.path.join(root, data_directory))

# Part A

## Exercise 2: Fast Feature Extraction vs. End-to-End Feature Extraction

When using a pretrained model, you can extract features in two ways: fast extraction (recording outputs to a NumPy array without data augmentation) and end-to-end extraction (chaining a frozen base with a new classifier to allow data augmentation)12,13.

-   **Dataset:** Oxford Flowers 102.-   **Task:** Instantiate the pretrained Xception backbone using `keras_hub.models.Backbone.from_preset("xception_41_imagenet")`14.-   **Part A (Fast Extraction):** Write a function that iterates over your dataset, passes images through the Xception backbone using `conv_base.predict()`, and records the outputs as a NumPy array15. Train a standalone densely connected classifier on these NumPy arrays16. Verify that training takes less than a second per epoch on a CPU17.-   **Part B (End-to-End Extraction):** Freeze the Xception base by setting `conv_base.trainable = False`18,19. Build a new `keras.Model` that chains your data augmentation stage, the frozen Xception base, and a Dense classification head20. Train this model end-to-end using a GPU.-   **Analysis:** Compare the final validation accuracy of Part A and Part B. Note how Part A severely overfits because it cannot utilize data augmentation21.

## Exercise 3: Precision Fine-Tuning

Fine-tuning adjusts the more abstract, specialized representations of a reused model to make them relevant to your specific problem22.

-   **Dataset:** Oxford Flowers 102.-   **Task:** Take the end-to-end model you built in Exercise 2, Part B.-   **Requirements:** Unfreeze **only the top 4 layers** of the Xception convolutional base using a loop (`for layer in conv_base.layers[:-4]: layer.trainable = False`)23. Recompile your model with a very low learning rate (e.g., `keras.optimizers.Adam(learning_rate=1e-5)`) to prevent large weight updates from destroying the previously learned representations23,24. Use `keras.callbacks.ModelCheckpoint` to monitor `val_loss` and save the best-performing model24.

## Exercise 4: Architectural Ablation (Max-Pooling vs. Strided Convolutions)

The purpose of max-pooling is to aggressively downsample feature maps to build a spatial hierarchy of features, usually utilizing 2x2 windows with a stride of 225.

-   **Dataset:** Dogs vs. Cats.-   **Task:** Build a ConvNet from scratch, but completely omit `MaxPooling2D` layers. Instead, perform spatial downsampling by passing `strides=2` into your `Conv2D` layers26,27.-   **Analysis:** Use `model.summary()` to compare the total number of trainable parameters between this strided-convolution model and a standard max-pooled model. Train both models and analyze whether relying on learned strided convolutions performs better or worse than the hardcoded max-tensor operation of `MaxPooling2D`25.